# Astramis — финальное обучение на всех данных, 50 эпох

YOLO11n-seg обучается на 263 реальных и 600 синтетических кадрах с разметкой. Все 137 кадров `real/test` обрабатываются после обучения. Результат сохраняется в `submission_50_epochs.csv`.

Финальный запуск использует все размеченные кадры. Параметр `val` в конфигурации Ultralytics указывает на обучающий набор для его штатной финальной диагностики. Для независимой оценки используйте исходный ноутбук с разбиением по физическим предметам.

In [1]:
# В Colab раскомментируйте следующую строку:
# !pip -q install ultralytics

# Во внутренней среде платформы эту строку:
# %pip install -q ultralytics opencv-python-headless pandas

In [2]:
import csv
import json
import os
import random
import shutil
from collections import defaultdict
from pathlib import Path

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO

BASE = Path("/home/jovyan/work/datasets")
cam = next(BASE.rglob("camera.json"), None)          # ищем, где реально лежат данные
DATA = cam.parent if cam else Path("dataset")        # локально и в Colab — как раньше

print("Данные:", DATA)

WORK = Path("work_50_epochs")                         # сюда пишем всё промежуточное
EPOCHS = 50                                 # эпох обучения детектора
IMGSZ = 640                                 # размер картинки на входе детектора
CONF = 0.10                                 # порог уверенности детектора
HEAVY_G = 500                               # тяжелее этого конвейер не берёт

DEVICE = "mps" if torch.backends.mps.is_available() else None

random.seed(0)
CAM = json.loads((DATA / "camera.json").read_text(encoding="utf-8"))


def color_path(frame_dir: Path) -> Path:
    """Путь к цветному кадру (png или jpg)."""
    p = frame_dir / "color.png"
    return p if p.exists() else frame_dir / "color.jpg"

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/jovyan/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Данные: /home/jovyan/work/datasets/dataset


## 1. Загружаем полную обучающую разметку

Объединяем реальные и синтетические кадры. Пустые реальные кадры остаются в обучении как примеры фона.

In [3]:
def load(labels_file):
    data = json.loads((DATA / "labels" / labels_file).read_text(encoding="utf-8"))
    anns = defaultdict(list)
    for a in data["annotations"]:
        anns[a["image_id"]].append(a)
    return [{"dir": DATA / Path(im["file_name"]).parent, "id": Path(im["file_name"]).parent.name,
             "w": im["width"], "h": im["height"], "anns": anns[im["id"]]}
            for im in data["images"]]


real = load("real_train.json")
synth = load("synth_train.json")

train_real = real
train = train_real + synth

cls_name = lambda a: f"{a['attributes']['form']}_{a['attributes']['material']}"
CLASSES = sorted({cls_name(a) for f in train for a in f["anns"]})
print(f"Обучение: {len(train_real)} реальных + {len(synth)} синтетических = {len(train)} кадров")
print("Классы:", CLASSES)

Обучение: 263 реальных + 600 синтетических = 863 кадров
Классы: ['bulk_metal', 'bulk_mineral', 'bulk_paper', 'bulk_plastic', 'bulk_wood', 'solid_glass', 'solid_metal', 'solid_mineral', 'solid_paper', 'solid_plastic', 'solid_wood']


## 2. Готовим данные для YOLO и обучаем детектор

Финальный прогон длится 50 эпох и использует веса последней эпохи (`last.pt`). На Mac с поддержкой MPS обучение идёт на Apple GPU.

In [4]:
yolo_dir = WORK / "yolo"
if yolo_dir.exists():
    shutil.rmtree(yolo_dir)
(yolo_dir / "images" / "train").mkdir(parents=True)
(yolo_dir / "labels" / "train").mkdir(parents=True)
for f in train:
    src = color_path(f["dir"])
    shutil.copy(src, yolo_dir / "images" / "train" / f"{f['id']}{src.suffix}")
    lines = []
    for a in f["anns"]:
        if cls_name(a) not in CLASSES:
            continue
        seg = max(a["segmentation"], key=len)            # самый большой контур объекта
        pts = np.array(seg, dtype=float).reshape(-1, 2) / [f["w"], f["h"]]
        lines.append(f"{CLASSES.index(cls_name(a))} " + " ".join(f"{v:.5f}" for v in pts.clip(0, 1).reshape(-1)))
    (yolo_dir / "labels" / "train" / f"{f['id']}.txt").write_text("\n".join(lines))

(yolo_dir / "data.yaml").write_text(
    f"path: {yolo_dir.resolve()}\ntrain: images/train\nval: images/train\nnames:\n"
    + "".join(f"  {i}: {n}\n" for i, n in enumerate(CLASSES)))

278

In [5]:
model = YOLO("yolo11n-seg.pt")
model.train(
    data=str(yolo_dir / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    device=DEVICE,
    batch=8 if DEVICE == "mps" else 16,
    val=False,
    patience=0,
    project=str(WORK.resolve()),
    name="seg",
    plots=False,
    verbose=False,
)
model = YOLO(str(model.trainer.last))

Ultralytics 8.4.216 🚀 Python-3.13.15 torch-2.14.0+cu126 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=work_50_epochs/yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=seg, nbs=64, nms=None, opset=None, optimize=False, opti

## 3. Объём по карте глубины

Каждый пиксель внутри маски переводим в точку 3D и считаем её высоту над плоскостью пола (плоскость записана в `meta.json` кадра).
Пол делим на клетки 4×4 мм, в каждой берём самую высокую точку, и складываем высоты, умноженные на площадь клетки.

In [6]:
def raw_volume_ml(depth_mm, mask, plane, cell=0.004):
    v, u = np.nonzero(mask & (depth_mm > 0))
    if len(v) < 30:
        return None                                          # глубины почти нет (стекло, блестящий металл)
    z = depth_mm[v, u] / 1000.0
    pts = np.stack([(u - CAM["cx"]) / CAM["fx"] * z, (v - CAM["cy"]) / CAM["fy"] * z, z], axis=1)
    n, d = np.array(plane[:3]), plane[3]
    height = pts @ n + d
    e1 = np.cross(n, [1.0, 0.0, 0.0])
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(n, e1)
    keys = np.floor(pts @ e1 / cell).astype(np.int64) * 100000 + np.floor(pts @ e2 / cell).astype(np.int64)
    order = np.argsort(keys)
    keys, height = keys[order], height[order]
    starts = np.r_[0, np.nonzero(np.diff(keys))[0] + 1]
    tops = np.maximum.reduceat(height, starts)
    return float(np.clip(tops, 0, None).sum() * cell * cell * 1e6)


def polygon_mask(segmentation, h, w):
    m = np.zeros((h, w), np.uint8)
    for seg in segmentation:
        cv2.fillPoly(m, [np.array(seg, np.float32).reshape(-1, 2).round().astype(np.int32)], 1)
    return m > 0


def read_frame(frame_dir):
    depth = cv2.imread(str(frame_dir / "depth.png"), cv2.IMREAD_UNCHANGED)
    plane = json.loads((frame_dir / "meta.json").read_text(encoding="utf-8"))["floor_plane"]
    return depth, plane

## 4. Поправочные коэффициенты и запасные значения

На обучающих реальных кадрах считаем «сырой» объём по настоящей маске и сравниваем с объёмом из разметки.
Медиана отношения внутри класса и есть поправка. Для объектов без глубины берём типичный объём класса.

In [7]:
ratios, volumes = defaultdict(list), defaultdict(list)
for f in train_real:
    if not f["anns"]:
        continue
    depth, plane = read_frame(f["dir"])
    for a in f["anns"]:
        true_v = a["attributes"]["volume_ml"]
        volumes[cls_name(a)].append(true_v)
        raw = raw_volume_ml(depth, polygon_mask(a["segmentation"], f["h"], f["w"]), plane)
        if raw and raw > 1:
            ratios[cls_name(a)].append(true_v / raw)

all_ratios = [r for v in ratios.values() for r in v]
FACTOR = {c: float(np.median(ratios[c])) if ratios[c] else float(np.median(all_ratios)) for c in CLASSES}
TYPICAL = {c: float(np.median(volumes[c])) for c in CLASSES}
with open(DATA / "densities.csv", encoding="utf-8") as fh:
    DENSITY = {f"{r['form']}_{r['material']}": float(r["density_g_per_ml"]) for r in csv.DictReader(fh)}
print(pd.DataFrame({"поправка": FACTOR, "типичный объём, мл": TYPICAL, "плотность, г/мл": DENSITY}).round(2))

               поправка  типичный объём, мл  плотность, г/мл
bulk_metal         1.68               240.0             1.46
bulk_mineral       0.86                45.0             1.64
bulk_paper         1.01              1050.0             0.01
bulk_plastic       0.85               130.0             0.57
bulk_wood          0.16                15.0             0.69
solid_glass        1.16               500.0             1.18
solid_metal        0.68               360.0             0.82
solid_mineral      0.88               100.0             1.60
solid_paper        0.94               800.0             0.39
solid_plastic      1.15              1000.0             0.28
solid_wood         0.32                40.0             0.88


## 5. Предсказание для одного кадра

In [8]:
def predict(frame_dir):
    """Возвращает список объектов кадра в формате метрики."""
    result = model(str(color_path(frame_dir)), conf=CONF, imgsz=IMGSZ, verbose=False)[0]
    if result.masks is None:
        return []
    depth, plane = read_frame(frame_dir)
    h, w = depth.shape
    out = []
    for box, conf, cls, poly in zip(result.boxes.xyxy.tolist(), result.boxes.conf.tolist(),
                                    result.boxes.cls.tolist(), result.masks.xy):
        name = CLASSES[int(cls)]
        form, material = name.split("_")
        volume = None
        if len(poly) >= 3:
            mask = np.zeros((h, w), np.uint8)
            cv2.fillPoly(mask, [poly.round().astype(np.int32)], 1)
            raw = raw_volume_ml(depth, mask > 0, plane)
            if raw and raw > 1:
                volume = raw * FACTOR[name]
        if volume is None:
            volume = TYPICAL[name]
        mass = volume * DENSITY.get(name, 1.0)
        x1, y1, x2, y2 = box
        if x2 - x1 < 2 or y2 - y1 < 2:
            continue
        out.append({"box": (x1, y1, x2, y2), "score": conf, "form": form, "material": material,
                    "volume": volume, "mass": mass, "take": int(mass <= HEAVY_G)})
    return out


def to_string(objs):
    """Список объектов -> строка для колонки prediction."""
    if not objs:
        return "none"
    return ";".join(f"{o['box'][0]:.1f} {o['box'][1]:.1f} {o['box'][2]:.1f} {o['box'][3]:.1f} {o['score']:.3f} "
                    f"{o['form']} {o['material']} {o['volume']:.0f} {o['mass']:.0f} {o['take']}" for o in objs)

## 6. Создаём файл для отправки

Предсказания для 137 тестовых кадров сохраняются в `submission_50_epochs.csv`.

In [9]:
test_dirs = sorted(p for p in (DATA / "real" / "test").iterdir() if p.is_dir())
rows = [{"image_id": d.name, "prediction": to_string(predict(d))} for d in test_dirs]
pd.DataFrame(rows).to_csv("submission_50_epochs.csv", index=False)
print(f"Готово: submission_50_epochs.csv, кадров {len(rows)}, с найденными объектами {sum(r['prediction'] != 'none' for r in rows)}")

Готово: submission_50_epochs.csv, кадров 137, с найденными объектами 114


## Что можно улучшить

- **Детектор.** Больше эпох, модель крупнее (`yolo11s-seg.pt`), другая доля синтетики, аугментации.
- **Материал.** Сейчас детектор определяет его заодно с формой. Можно обучить отдельный классификатор на вырезках объектов.
- **Глубина в детекции.** Детектор смотрит только на цвет. Высота над полом могла бы стать четвёртым каналом или отсеивать ложные срабатывания.
- **Объём.** Одна поправка на класс очень грубая. Попробуйте учесть площадь объекта, его высоту, расстояние до камеры, или обучить на этих признаках небольшую модель.
- **Стекло и блестящий металл.** У них в глубине дыры. Что делать, когда глубины нет?
- **Решение «забрать / объехать».** Массу через плотность класса не угадать для полной и пустой бутылки. Можно ли предсказывать решение напрямую?
- **Анализ ошибок.** В исходном ноутбуке с валидацией выведите кадры, где модель ошиблась сильнее всего. Это пригодится на защите.